In [ ]:
# ============================================================
# THERMAL WIND PREDICTION PROJECT
# BUOY DATA PREPROCESSING
# ============================================================

# -------------------------
# IMPORTS
# -------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Configuración visual
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries loaded successfully.")

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

# Ajusta esta ruta a tu proyecto
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

# Crear carpetas si no existen
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project paths ready.")
print(f"RAW DATA: {RAW_DATA_PATH}")
print(f"PROCESSED DATA: {PROCESSED_DATA_PATH}")

In [ ]:
# ============================================================
# LOAD BUOY DATASET
# ============================================================

file_name = "Boya 2005-2025 - 25933_53782_1731_ALL_20050101120619_20251231120619.csv"

file_path = RAW_DATA_PATH / file_name

# Leer CSV
df = pd.read_csv(
    file_path,
    sep="\t",
    encoding="latin1"
)

print("Dataset loaded successfully.")

In [ ]:
# ============================================================
# INITIAL INSPECTION
# ============================================================

print("\nDATAFRAME SHAPE")
print(df.shape)

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nFIRST ROWS")
display(df.head())

print("\nINFO")
df.info()

In [ ]:
# ============================================================
# RENAME COLUMNS
# ============================================================

# IMPORTANTE:
# Ajusta los nombres EXACTOS según aparezcan en tu CSV

df = df.rename(columns={

    'Fecha': 'datetime',

    'Temperatura del Agua(ºC)': 'sst',

    'Altura Signif. del Oleaje (Hm0)(m)': 'wave_height',

    'Periodo Medio(s)': 'wave_period_mean',

    'Periodo Medio Tm02(s)': 'wave_period_tm02',

    'Periodo de Pico(s)': 'wave_peak_period',

    'Direcc. Media de Proced.(0=N,90=E)': 'wave_direction'

})

print("Columns renamed.")

In [ ]:
# ============================================================
# CONVERT MISSING VALUES
# ============================================================

# Reemplazar valores inválidos típicos
invalid_values = [-9999, -9999.0, -9999.9, -9999.99]

df.replace(invalid_values, np.nan, inplace=True)

print("Invalid values converted to NaN.")

In [ ]:
# ============================================================
# DATETIME PARSING
# ============================================================

# Ver cómo viene originalmente
display(df['datetime'].head())

# Parsear datetime
df['datetime'] = pd.to_datetime(
    df['datetime'],
    format='%Y %m %d %H',
    errors='coerce'
)

print("Datetime converted.")

In [ ]:
# ============================================================
# REMOVE INVALID DATETIMES
# ============================================================

before_rows = len(df)

df = df.dropna(subset=['datetime'])

after_rows = len(df)

print(f"Removed rows with invalid datetime: {before_rows - after_rows}")

In [ ]:
# ============================================================
# SET DATETIME INDEX
# ============================================================

df = df.set_index('datetime')

# Ordenar temporalmente
df = df.sort_index()

print("Datetime index configured.")

In [ ]:
# ============================================================
# REMOVE DUPLICATES
# ============================================================

duplicates = df.index.duplicated().sum()

print(f"Duplicated timestamps: {duplicates}")

# Eliminar duplicados
df = df[~df.index.duplicated(keep='first')]

print("Duplicates removed.")

In [ ]:
# ============================================================
# SELECT RELEVANT COLUMNS
# ============================================================

selected_columns = [

    'sst',
    'wave_height',
    'wave_period_mean',
    'wave_period_tm02',
    'wave_peak_period',
    'wave_direction'

]

df = df[selected_columns]

print("Relevant columns selected.")
display(df.head())

In [ ]:
# ============================================================
# CHECK MISSING VALUES
# ============================================================

missing_values = df.isna().sum().sort_values(ascending=False)

missing_percentage = (missing_values / len(df)) * 100

missing_report = pd.DataFrame({
    'missing_values': missing_values,
    'missing_percentage': missing_percentage
})

display(missing_report)

In [ ]:
# ============================================================
# INTERPOLATE SST
# ============================================================

# SOLO SST
# Porque la temperatura del mar cambia lentamente

df['sst'] = df['sst'].interpolate(
    method='time',
    limit=24
)

print("SST interpolation completed.")

In [ ]:
# ============================================================
# BASIC SST QUALITY CHECK
# ============================================================

print("\nSST STATISTICS")
display(df['sst'].describe())

print("\nSST MIN/MAX")
print(f"Min SST: {df['sst'].min():.2f} °C")
print(f"Max SST: {df['sst'].max():.2f} °C")

In [ ]:
# ============================================================
# SST TIME SERIES PLOT
# ============================================================

plt.figure(figsize=(18, 6))

plt.plot(df.index, df['sst'])

plt.title('Sea Surface Temperature (SST) - Barcelona Buoy')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# MONTHLY SST CLIMATOLOGY
# ============================================================

monthly_sst = df['sst'].groupby(df.index.month).mean()

plt.figure(figsize=(12, 5))

monthly_sst.plot(marker='o')

plt.title('Monthly Mean SST Climatology')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# TEMPORAL CONTINUITY CHECK
# ============================================================

# Diferencias temporales
time_diff = df.index.to_series().diff()

print(time_diff.value_counts().head(10))

In [ ]:
# ============================================================
# CREATE BASIC SST FEATURES
# ============================================================

# Rolling means
df['sst_roll_6h'] = df['sst'].rolling(6).mean()

df['sst_roll_24h'] = df['sst'].rolling(24).mean()

# SST changes
df['sst_delta_3h'] = df['sst'].diff(3)

df['sst_delta_24h'] = df['sst'].diff(24)

print("Basic SST features created.")

In [ ]:
# ============================================================
# FINAL INSPECTION
# ============================================================

display(df.head())

print(df.info())

In [ ]:
# ============================================================
# SAVE CLEAN DATASET
# ============================================================

output_file = PROCESSED_DATA_PATH / "barcelona_buoy_clean.parquet"

df.to_parquet(output_file)

print("Clean dataset saved successfully.")
print(output_file)

In [ ]:
# ============================================================
# OPTIONAL CSV EXPORT
# ============================================================

csv_output = PROCESSED_DATA_PATH / "barcelona_buoy_clean.csv"

df.to_csv(csv_output)

print("CSV export completed.")